In [1]:
%load_ext rpy2.ipython

In [2]:
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import functools as ft
import clean_eicv_script as ces
from functools import reduce

### 

In [4]:
%%R

# Source of Code: Gabriel Sutil (eicv5 cleaning)

# install.packages("pacman")
pacman::p_load('data.table','dplyr','plyr','haven','mgcv','stargazer')

# Source of Code: Gabriel Sutil (EICV5 cleaning)
script_path <- '/home/qsel/Desktop/My_Drive/.shared/rwanda_drive/elec_consumption/code/clean_eicv_scripts.R'
# script_path <- '/Users/joelmugyenyi/Google Drive/My Drive/rwanda_drive/elec_consumption/code/clean_eicv_scripts.R'
source(script_path)

path_dir <- '/home/qsel/Desktop/data/REG_electricity_expenditure/' # <--change path to eicv5 data directory
# path_dir <- '/Users/joelmugyenyi/Google Drive/My Drive/rwanda_drive/elec_consumption/data/' # <--change path to eicv5 data directory
setwd(path_dir)

# Import Section ---------------------------------------------------------------
eicv5 <- read_sav('eicv5/cs_S1_S2_S3_S4_S6A_S6E_Person.sav')
demographics_r <- clean_demographics(eicv5)

section5 <- read_sav('eicv5/cs_S0_S5_Household.sav')
hh_characteritics_r <- clean_household_characteristics(section5)
rent_r <- clean_rent(section5) # Rent expenditure 
water_r <- clean_water(section5) # Rent expenditure 
electricity_r <- clean_electricity(section5) # Rent expenditure zs
cooking_r <- clean_cooking(section5) # Rent expenditure 


# Savings -------------------------------------------------
section10C1 <- read_sav('eicv5/cs_S10C1_Savings.sav')
savings_r<- clean_savings(section10C1)

# Credit -------------------------------------------------
section10A2 <- read_sav('eicv5/cs_S10A2_Listing_of_credits.sav')
loans_r<- clean_loans(section10A2)

# Durable goods PART A  -------------------------------------------------
section10B1 <- read_sav('eicv5/cs_S10B1_Durable_household_goods.sav')
furniture_r <- clean_furniture(section10B1)

# Durable goods PART B  -------------------------------------------------
section10B2 <- read_sav('eicv5/cs_S10B2_Durable_household_goods.sav')
appliances_r <- clean_appliances(section10B2)

# import poverty
poverty_r <- read_sav('eicv5/EICV5_Poverty_file.sav')

In [6]:
%%R
# Annual expenditure -------------------------------------------------
options(dplyr.summarise.inform = FALSE)
exp <- read_sav('eicv5/cs_S8A1_expenditure.sav')
expenditure_r <- clean_expenditure(exp)

In [5]:
%R -o furniture_r -o appliances_r -o demographics_r -o poverty_r -o rent_r #-o expenditure_r

In [6]:
%R -o loans_r -o savings_r -o hh_characteritics_r -o electricity_r -o water_r -o cooking_r 

In [8]:
### Clean data
# Demographics -------
# import demographics file in python
dir_path = '/home/qsel/Desktop/data/REG_electricity_expenditure/eicv5/'
# dir_path = '/Users/joelmugyenyi/Google Drive/My Drive/rwanda_drive/elec_consumption/data/eicv5/'

demo = pd.read_spss(dir_path+'cs_S1_S2_S3_S4_S6A_S6E_Person.sav') 
hhids = demo.groupby('hhid')['pid'].count().reset_index()
tmp1 = demo[['hhid','pid','s4bq8a']]
tmp2 = tmp1[tmp1.s4bq8a=='Yes'].groupby('hhid')['pid'].count().reset_index().rename(columns={'pid':'computer'})
computer = hhids.merge(tmp2,how='left').fillna(0)[['hhid','computer']]
computer['computer_present'] = computer.computer.apply(lambda x: 1 if x>0 else x)

tmp3 = demo[['hhid','pid','s4bq7']]
tmp4 = tmp3[tmp3.s4bq7=='Yes'].groupby('hhid')['pid'].count().reset_index().rename(columns={'pid':'Mobile_phones'})
phones = hhids.merge(tmp4,how='left').fillna(0)[['hhid','Mobile_phones']]
phones['Mobile_phones_present'] = phones.Mobile_phones.apply(lambda x: 1 if x>0 else x)

tmp5 = demo[['hhid','pid','s4bq7a']]
tmp6 = tmp5[(tmp5.s4bq7a=='Yes, internet only')|(tmp5.s4bq7a=='Yes, radio and internet')].groupby('hhid')['pid'].count().reset_index().rename(columns={'pid':'smart_phones'})
smart_phones = hhids.merge(tmp6,how='left').fillna(0)[['hhid','smart_phones']]


school_registration = ces.school_costs(demo,'s4aq11a','school_registration_fees')
parent_contributions = ces.school_costs(demo,'s4aq11b','parent_contributions')
school_uniforms = ces.school_costs(demo,'s4aq11c','school_uniforms')
school_supplies = ces.school_costs(demo,'s4aq11d','school_supplies')
school_transport = ces.school_costs(demo,'s4aq11e','school_transport')
school_room_board = ces.school_costs(demo,'s4aq11f','school_room_board')
other_school_costs = ces.school_costs(demo,'s4aq11g','other_school_costs')
total_educ_costs = ces.school_costs(demo,'s4aq11h','total_educ_costs')
vocational_costs = ces.school_costs(demo,'s4bq2','vocational_costs')

demographics_r.district = demographics_r.district.replace(ces.district_mapping)
demographics_r.province = demographics_r.province.replace(ces.province_mapping)
demographics_r.urban_rural = demographics_r.urban_rural.replace(ces.urban_rural_mapping) 
demographics_r = pd.concat((demographics_r,pd.get_dummies(demographics_r.urban_rural)),axis=1)

dfs = [
    demographics_r,school_registration,parent_contributions,school_uniforms,
    school_supplies,school_transport,school_room_board,other_school_costs,total_educ_costs,vocational_costs
      ]
demographics = ft.reduce(lambda left, right: pd.merge(left, right, on='hhid'), dfs)



# Appliances-------
appliance_count = ces.filter_appliance(appliances_r)
appliance_count = appliance_count.drop(['computer','Mobile_phones',
                                        'num_appliances_HH'],axis=1).merge(computer[
    ['hhid','computer']],how='left').merge(phones[['hhid','Mobile_phones']],how='left').merge(smart_phones[['hhid','smart_phones']],how='left')


# Annual expenses ------------
poverty = ces.filter_poverty(poverty_r)


# Non electric articles ----------------------
non_electric_count =  ces.filter_non_electric(furniture_r,appliances_r)

# appliance presence
appliance_presence = appliance_count.drop(columns=['hhid']).clip(0,1)
appliance_presence.insert(0, 'hhid', appliance_count.hhid)

#### Appliance tiers

In [10]:
## Categorization appliances
tier_0_appliances = [ 'Radio', 'Mobile_phones', 'TV', 'satellite_dish', 'dvd_player',
       'decoder', 'music_syst', 'computer', 'cooker', 'laundry_mach',
       'electric_fan', 'fridge', 'hotplate', 'still_camera', 'video_camera',
       'printer']
# Charging
tier_1_appliances = ['Radio', 'Mobile_phones']
# Entertainment
tier_2_appliances = ['TV', 'satellite_dish', 'dvd_player','decoder','music_syst',]
# Productivity
tier_3_appliances = ['computer','printer']
# Heavy duty
tier_4_appliances = [ 'cooker', 'laundry_mach','fridge', 'hotplate', 'electric_fan']
tier_list = [tier_1_appliances, tier_2_appliances, tier_3_appliances, tier_4_appliances]

## Condtions
any_tier1_appliance = (appliance_presence[tier_1_appliances]==1).any(axis=1)
any_tier2_appliance = (appliance_presence[tier_2_appliances]==1).any(axis=1)
any_tier3_appliance = (appliance_presence[tier_3_appliances]==1).any(axis=1)
any_tier4_appliance = (appliance_presence[tier_4_appliances]==1).any(axis=1)


# assignment to appliance tiers
tier_0_hhids = appliance_presence[
    (appliance_presence[tier_0_appliances]==0).all(axis=1)]['hhid'].tolist()
strict_list = ces.strict_tier_assignment(appliance_presence, any_tier1_appliance,any_tier2_appliance,
                           any_tier3_appliance,any_tier4_appliance)
assert ces.check_list_unique(strict_list) == 0 # check fot tier duplicates
appliance_presence['tier_level'] = appliance_presence.apply( lambda x: ces.strict_assign_tier(x['hhid'],
 tier_0_hhids,strict_list[0], strict_list[1],strict_list[2], strict_list[3]),axis=1)
loose_list = ces.loose_tier_assignment(appliance_presence,any_tier2_appliance,
                           any_tier3_appliance,any_tier4_appliance)
appliance_presence['tier_level'] = appliance_presence.apply( lambda x: ces.loose_assign_tier(x['hhid'],
 x['tier_level'],loose_list[0], loose_list[1],loose_list[2]),axis=1)

appliance_presence_HH = appliance_presence.add_suffix('_present')
appliance_presence_HH = appliance_presence_HH.rename(columns={'hhid_present':'hhid','tier_level_present':'tier_level'})
appliance_presence_HH['tier_level_numeric'] = appliance_presence_HH.tier_level.apply(lambda x: ces.convert_tier_level_2_numeric(x))
appliance_presence_HH = pd.concat((appliance_presence_HH,pd.get_dummies(appliance_presence_HH.tier_level)),axis=1)


### Household Salaries

In [37]:
### Clean data
# Employment -------
# import demographics file in python
# dir_path = '/home/qsel/Desktop/data/REG_electricity_expenditure/eicv5/'
dir_path = '/Users/joelmugyenyi/Google Drive/My Drive/rwanda_drive/elec_consumption/data/eicv5/'
demo = pd.read_spss(dir_path+'cs_S1_S2_S3_S4_S6A_S6E_Person.sav') 
hhids = demo.groupby('hhid')['pid'].count().reset_index()

employ = pd.read_spss(dir_path+'cs_S6B_Employement_6C_Salaried_S6D_Business.sav') 

# get Household salary information
HH_salaries = ces.get_salary_info(employ)

# businesses
turnover = employ[['hhid','pid','s6dq34b','s6dq34a']].dropna().pivot_table(index='hhid',values='s6dq34a',columns='s6dq34b').rename(columns={
    'Annually':'annual_turnover_biz',
    'Daily':'daily_turnover_biz',
    'Monthly':'monthly_turnover_biz',
    'Weekly':'weekly_turnover_biz',
    }).fillna(0).reset_index()
# turnover = turnover.div(turnover.sum(axis=1), axis=0).reset_index()
turnover['received_income_from_business'] = 1

#merge
merge = hhids.merge(HH_salaries, how='left').merge(turnover,how='left').drop('pid',axis=1).fillna(0)
turnover_dummies = pd.get_dummies(merge.received_income_from_business).rename(columns={0:'no_business_turnover',1:'yes_business_turnover'})
HH_salary_and_business=pd.concat([merge,turnover_dummies],axis=1)
HH_salary_and_business.shape

(14580, 17)

In [94]:
HH_salary_and_business = HH_salary_and_business.merge(hh_wage,how='left')

In [92]:
hh_wage = employ[['hhid','s6cq17a']].groupby('hhid')['s6cq17a'].sum().clip(0,1).reset_index().rename(columns={
    's6cq17a':'HH_wage'
})
hh_wage

,hhid,HH_wage
0,200001.0,1.0
1,200002.0,0.0
2,200003.0,1.0
3,200004.0,1.0
4,200005.0,1.0
...,...,...
14575,214576.0,1.0
14576,214577.0,0.0
14577,214578.0,0.0
14578,214579.0,0.0


### VUP Loans scheme, loans, savings

In [25]:
### Clean data
# Employment -------
# import demographics file in python
# dir_path = '/home/qsel/Desktop/data/REG_electricity_expenditure/eicv5/'
dir_path = '/Users/joelmugyenyi/Google Drive/My Drive/rwanda_drive/elec_consumption/data/eicv5/'

# loans
loans_and_debt = pd.read_spss(dir_path+'cs_S10A1_Credits.sav') 
loans = pd.get_dummies(loans_and_debt.s10aq1).rename(columns={'No':'No_attempted_loan','Yes':'Yes_attempted_loan'})
debt = pd.get_dummies(loans_and_debt.s10aq3).rename(columns={'No':'No_debt','Yes':'Yes_debt'})
HH_loan_and_debt = pd.concat([loans_and_debt['hhid'],loans,debt],axis=1)

loan_amount = pd.read_spss(dir_path+'cs_S10A2_Listing_of_credits.sav') 
loan_amount = loan_amount[['hhid','s10aq7']].groupby('hhid')['s10aq7'].sum().reset_index().rename(columns={
    's10aq7':'loan_amount',
})
HH_loan_and_debt = HH_loan_and_debt.merge(loan_amount, how='left').fillna(0)


# savings
savings = pd.read_spss(dir_path+'cs_S10C1_Savings.sav') 
savings = savings.groupby('hhid')['s10cq6'].sum().reset_index()
HH_savings = hhids.merge(savings, how='left')[['hhid','s10cq6']].fillna(0).rename(columns={
    's10cq6':'total_savings'
})



# vup
vup_ds_tmp = pd.read_spss(dir_path+'cs_S9C_Vup_ubudehe_and_Rssp_schemes.sav') 

vup_ds=vup_ds_tmp[['hhid','s9cq1','s9cq2','s9cq4a','s9cq4b','s9cq4c','s9cq4d','s9cq4e','s9cq4f','s9cq4g','s9cq4h','s9cq4i','s9cq4j','s9cq8','s9cq11',
     's9cq13a','s9cq13b','s9cq13c','s9cq13d','s9cq13e','s9cq13f','s9cq13g','s9cq13h','s9cq13i','s9cq13j','s9cq13k','s9cq13l','s9c5q45','s9c5q46a','s9c5q46b']].rename(columns={
    's9cq1':'vup_HH_category','s9cq2':'received_income_vup',
's9cq4a':'programs_2008','s9cq4b':'programs_2009','s9cq4c':'programs_2010','s9cq4d':'programs_2011','s9cq4e':'programs_2012','s9cq4f':'programs_2013',
's9cq4g':'programs_2014','s9cq4h':'programs_2015','s9cq4i':'programs_2016','s9cq4j':'programs_2017','s9cq8':'avg_amount_last_ds',
's9cq11':'ds_amount_last_12_months',
     's9cq13a':'invest_food','s9cq13b':'invest_clothes','s9cq13c':'invest_utensils','s9cq13d':'invest_durable_assets','s9cq13e':'invest_education','s9cq13f':'invest_health',
     's9cq13g':'invest_animals','s9cq13h':'invest_farm','s9cq13i':'invest_business','s9cq13j':'invest_home_improvements','s9cq13k':'invest_savings',
     's9cq13l':'invest_others','s9c5q45':'Rssp_receipient','s9c5q46a':'rssp_coop','s9c5q46b':'invest_access_rif_credit'
     }).copy()
vup_ds.received_income_vup = vup_ds.received_income_vup.apply(lambda row: ces.categorize_received_vup(row))
vup_ds = hhids.merge(vup_ds, how='left')
vup_ds.received_income_vup = vup_ds.received_income_vup.fillna('No_VUP')
dummies_income = pd.get_dummies(vup_ds.received_income_vup)
dummies_cats = pd.get_dummies(vup_ds.vup_HH_category).rename(columns={
    'Category 1':'vup_Category_1',
    'Category 2':'vup_Category_2',
    'Category 3':'vup_Category_3',
    'Category 4':'vup_Category_4',
    'Don?t know':'vup_dont_know'
})
priority_cols = ['hhid','No_VUP','Yes_VUP']
vup = pd.concat([vup_ds[['hhid']],dummies_income,dummies_cats],axis=1)[priority_cols]

# merge
HH_credit_savings = vup.merge(HH_loan_and_debt, how='left').merge(HH_savings, how='left')
HH_credit_savings


,hhid,No_VUP,Yes_VUP,No_attempted_loan,Yes_attempted_loan,No_debt,Yes_debt,loan_amount,total_savings
0,200001.0,1,0,1,0,1,0,0.0,40000.0
1,200002.0,1,0,1,0,1,0,0.0,20000.0
2,200003.0,1,0,1,0,1,0,0.0,50000.0
3,200004.0,1,0,1,0,0,1,12000.0,4000.0
4,200005.0,1,0,1,0,0,1,3150000.0,434000.0
...,...,...,...,...,...,...,...,...,...
14575,214576.0,1,0,1,0,0,1,100000.0,55500.0
14576,214577.0,0,1,1,0,0,1,550000.0,281200.0
14577,214578.0,1,0,1,0,0,1,3000.0,0.0
14578,214579.0,1,0,1,0,0,1,10000000.0,800000.0


### Income Agriculture

In [7]:
### Clean data
# Employment -------
# import demographics file in python
dir_path = '/home/qsel/Desktop/data/REG_electricity_expenditure/eicv5/'
# dir_path = '/Users/joelmugyenyi/Google Drive/My Drive/rwanda_drive/elec_consumption/data/eicv5/'

agric_income = pd.read_spss(dir_path+'cs_S7F_income_agriculture.sav') 
agric_income_amounts = agric_income.groupby(['hhid'])[['s7fq2','s7fq3']].sum().reset_index().rename(columns={
    's7fq2':'agric_income_4_wks',
    's7fq3':'agric_income_12_months'
})

x = agric_income.groupby(['hhid','s7fq1'])['s7fqcode'].count().reset_index()
agric_involvement = x[x.s7fqcode!=0][['hhid','s7fq1']].rename(columns={'s7fq1':'obtained_agric_income'}).drop_duplicates(subset=['hhid'],keep='last')
agric_involvement.shape

(14578, 2)

small and large crop agriculture

In [136]:
demo = pd.read_spss(dir_path+'cs_S1_S2_S3_S4_S6A_S6E_Person.sav') 
hhids = demo.groupby('hhid')['pid'].count().reset_index()
# small crop
small_crop = pd.read_spss(dir_path+'cs_S7E_small_crop.sav') 
small_crop.s7eq5 = small_crop.s7eq5.apply(lambda row: ces.categorize_small_crop_sales(row))
small_crop= small_crop.groupby('hhid')[['s7eq8']].sum().reset_index().rename(columns={'s7eq8':'small_crop_annual_sale'})
small_crop['has_small_crop_agric'] = small_crop.small_crop_annual_sale.apply(lambda x: 1 if (x>0) else x )
small_crops_HH = hhids.merge(small_crop,how='left')[['hhid','has_small_crop_agric']].fillna(0)
small_crop_presence = pd.get_dummies(small_crops_HH.has_small_crop_agric).rename(columns={0:'no_small_crops',1:'yes_small_crops'})
HH_small_crops = pd.concat([small_crops_HH['hhid'],small_crop_presence],axis=1)

# large crop
large_crop = pd.read_spss(dir_path+'cs_S7D_large_crop.sav') 
large_crop = large_crop.groupby('hhid')[['s7dq7']].sum().reset_index().rename(columns={'s7dq7':'large_crop_annual_sale'})
large_crop['has_large_crop_agric'] = large_crop.large_crop_annual_sale.apply(lambda x: 1 if (x>0) else x )
large_crop_HH = hhids.merge(large_crop,how='left')[['hhid','has_large_crop_agric']].fillna(0)
large_crop_presence = pd.get_dummies(large_crop_HH.has_large_crop_agric).rename(columns={0:'no_large_crops',1:'yes_large_crops'})
HH_large_crop = pd.concat([large_crop_HH['hhid'],large_crop_presence],axis=1)
HH_crop_agriculture = HH_small_crops.merge(HH_large_crop)
HH_crop_agriculture.shape

(14580, 5)

### Money Transfers

In [222]:
# load transfer data
transfers = pd.read_spss(dir_path+'cs_S9B_transfers_in.sav') 
transfers = transfers[['hhid','s9bq0','s9bq2','s9bq5','s9bq6','s9bq7','s9bq8','s9bq9','s9bq10']]
transfers.s9bq7 = pd.to_numeric(transfers.s9bq7, errors='coerce')

# source of transfers by percentage
transfers['total_transfer_amount'] = transfers[['s9bq7','s9bq10']].sum(axis=1)
transfers_pvt = transfers.pivot_table(index='hhid',values='total_transfer_amount',columns='s9bq5').fillna(0)
transfers_pct = transfers_pvt.div(transfers_pvt.sum(axis=1), axis=0).reset_index().rename(columns={
    'Big town':'big_town_pct',
    'Other African countries':'other_african_countries_pct',
    'Other countryside':'other_countryside_pct',
    'Other town':'other_town_pct',
    'Out of Africa':'out_of_africa_pct',
    'Same village or town':'same_village_or_town_pct',
    'bordering countries':'bordering_countries_pct',
    'Kigali':'kigali_pct'
})
# HH's that received a transfer
y = transfers.groupby(['hhid','s9bq6'])['s9bq7'].sum().reset_index() 
cash_transfers_tmp = y[~((y.s9bq6=='Yes')&(y.s9bq7==0))].rename(columns={'s9bq6':'cash_sender','s9bq7':'cash_received_12_months'}).drop_duplicates(subset='hhid',keep='last')
cash_dummies = pd.get_dummies(cash_transfers_tmp.cash_sender).rename(columns={'No':'no_cash_transfers','Yes':'yes_cash_transfers'})
cash_transfers = pd.concat([cash_transfers_tmp[['hhid','cash_received_12_months']],cash_dummies],axis=1)

y = transfers.groupby(['hhid','s9bq9'])['s9bq10'].sum().reset_index() 
food_transfers_tmp = y[~((y.s9bq9=='Yes')&(y.s9bq10==0))].rename(columns={'s9bq9':'food_sender','s9bq10':'food_received_12_months'}).drop_duplicates(subset='hhid',keep='last')
food_dummies = pd.get_dummies(food_transfers_tmp.food_sender).rename(columns={'No':'no_food_transfers','Yes':'yes_food_transfers'})
food_transfers = pd.concat([food_transfers_tmp[['hhid','food_received_12_months']],food_dummies],axis=1)

# merging

food_cash_transfers = food_transfers.merge(cash_transfers,how='outer').merge(transfers_pct,how='left').fillna(0)
food_cash_transfers.shape

(14575, 15)

### services

In [444]:
demo = pd.read_spss(dir_path+'cs_S1_S2_S3_S4_S6A_S6E_Person.sav') 
tmp7 = demo[['hhid','pid','s4bq9k','s4bq9l','s4bq9m','s4bq9n']].copy()

# internet use 
tmp7['internet_last_3_months'] = (tmp7[['s4bq9k','s4bq9l','s4bq9m','s4bq9n']] == 'Yes').any(axis=1).apply(lambda x: 1 if x==True else 0)
tmp8 = tmp7.groupby('hhid')['internet_last_3_months'].sum().clip(0,1).reset_index()
tmp9 = pd.get_dummies(tmp8.internet_last_3_months).rename(columns={0:'not_used_internet_last_3_mnths',1:'used_internet_last_3_mnths'})
internet_use = pd.concat([tmp8['hhid'],tmp9],axis=1)

# Agric involvement
tmp11 = demo[['hhid','pid','s6aq3']].copy()
tmp12 = tmp11[tmp11.s6aq3=='Yes'].replace({'Yes':1}).drop_duplicates(subset='hhid',keep='last')
tmp12['s6aq3'] = pd.to_numeric(tmp12['s6aq3'], errors='coerce')
agric_activity = hhids.merge(tmp12[['hhid','s6aq3']], how='left')[['hhid','s6aq3']].fillna(0)
tmp13 = pd.get_dummies(agric_activity.s6aq3).rename(columns={0:'no_agric_activity',1:'yes_agric_activity'})
agric_activity = pd.concat([agric_activity['hhid'],tmp13],axis=1)

# non farm wages
tmp14 = demo[['hhid','pid','s6aq4']].copy()
tmp15 = tmp14[tmp14.s6aq4=='Yes'].replace({'Yes':1}).drop_duplicates(subset='hhid',keep='last')
tmp15['s6aq4'] = pd.to_numeric(tmp15['s6aq4'], errors='coerce')
wages_non_farm = hhids.merge(tmp15[['hhid','s6aq4']], how='left')[['hhid','s6aq4']].fillna(0)
tmp13 = pd.get_dummies(wages_non_farm.s6aq4).rename(columns={0:'no_non_farm_wage',1:'yes_non_farm_wage'})
wages_non_farm = pd.concat([wages_non_farm['hhid'],tmp13],axis=1)

# own non farm business
tmp14 = demo[['hhid','pid','s6aq5']].copy()
tmp15 = tmp14[tmp14.s6aq5=='Yes'].replace({'Yes':1}).drop_duplicates(subset='hhid',keep='last')
tmp15['s6aq5'] = pd.to_numeric(tmp15['s6aq5'], errors='coerce')
biz_non_farm = hhids.merge(tmp15[['hhid','s6aq5']], how='left')[['hhid','s6aq5']].fillna(0)
tmp13 = pd.get_dummies(biz_non_farm.s6aq5).rename(columns={0:'no_non_farm_biz',1:'yes_non_farm_biz'})
biz_non_farm = pd.concat([biz_non_farm['hhid'],tmp13],axis=1)
biz_non_farm[biz_non_farm.yes_non_farm_biz==1]

# job stability
tmp14 = demo[['hhid','pid','s6aq7a']].copy()
job_stability = tmp14.groupby('hhid')['s6aq7a'].mean().reset_index().rename(columns={'s6aq7a':'job_stability'})

HH_work_stats = reduce(lambda x, y: x.merge(y, on='hhid'), [internet_use,agric_activity,wages_non_farm,biz_non_farm,job_stability])
HH_work_stats

,hhid,not_used_internet_last_3_mnths,used_internet_last_3_mnths,no_agric_activity,yes_agric_activity,no_non_farm_wage,yes_non_farm_wage,no_non_farm_biz,yes_non_farm_biz,job_stability
0,200001.0,0,1,0,1,0,1,1,0,1.500000
1,200002.0,1,0,1,0,1,0,0,1,2.000000
2,200003.0,1,0,1,0,0,1,0,1,2.000000
3,200004.0,1,0,1,0,0,1,0,1,1.000000
4,200005.0,0,1,1,0,0,1,0,1,1.000000
...,...,...,...,...,...,...,...,...,...,...
14575,214576.0,1,0,0,1,0,1,1,0,2.666667
14576,214577.0,1,0,1,0,1,0,0,1,1.600000
14577,214578.0,1,0,1,0,1,0,0,1,2.000000
14578,214579.0,1,0,1,0,1,0,0,1,2.000000


In [113]:
services_orign = [
'Food market/ shop',
# 'Market for selling farm produce',
'Public transport stage',
'All-weather roads',
'Pre-primary',
'Primary schools',
'Secondary schools',
# 'District hospital',
'Health centre',
'Sector office',
# 'Internet services',
'Cellule office' 
]
services_renamed = [
'food_market_or_shop',
# 'mkt_2_sell_crops',
'transport_stage',
'all_weather_roads',
'pre_primary',
'primary',
'secondary',
# 'district_hospital',
'health_centre',
'sector_office',
# 'Internet_services', 
'Cellule_office'
]
svcs = pd.read_spss(dir_path+'cs_S5F_Access_to_services.sav')
df_list = []
for num in range(0,len(services_orign)):
    service = services_renamed[num]
    svcs[f'time_2_{service}'] = (svcs.s5fq3h)*60 + svcs.s5fq3m
    df = svcs[svcs.s5fq0==services_orign[num]].rename(columns={
        's5fq4':f'dist_2_{service}',
        's5fq5':f'transprt_2_{service}'
        })[['hhid',f'time_2_{service}',f'dist_2_{service}',f'transprt_2_{service}']]
    df[f'dist_2_{service}'] = pd.to_numeric(df[f'dist_2_{service}'],errors='coerce')/1000
    df_list.append(df)

df_svcs = reduce(lambda x, y: x.merge(y, on='hhid'), df_list)

transport_mode =['transprt_2_food_market_or_shop','transprt_2_transport_stage','transprt_2_all_weather_roads',
         'transprt_2_pre_primary','transprt_2_primary','transprt_2_secondary','transprt_2_health_centre',
         'transprt_2_sector_office','transprt_2_Cellule_office']
df_svcs['bicycle_access'] =(df_svcs[transport_mode]=='Bicycle').any(axis=1).apply(lambda x: 0 if x==False else 1)
df_svcs['motorcycle_access'] =(df_svcs[transport_mode]=='Motorcycle').any(axis=1).apply(lambda x: 0 if x==False else 1)
df_svcs['vehicle_access'] =(df_svcs[transport_mode]=='Vehicle').any(axis=1).apply(lambda x: 0 if x==False else 1)
df_svcs['minibus_access'] =(df_svcs[transport_mode]=='Minibus').any(axis=1).apply(lambda x: 0 if x==False else 1)
df_svcs = df_svcs.drop(transport_mode,axis=1)

# transport stage quality
service = 'transport_stage'
df = svcs[svcs.s5fq0=='Public transport stage'].rename(columns={'s5fq4':f'dist_2_{service}'})[['hhid','s5fq6']]
tmp = pd.get_dummies(df.s5fq6).rename(columns={'Not satisfied':'stage_unsatisfactory','Satisfied':'stage_satisfactory'})
stage_quality = pd.concat([df['hhid'],tmp],axis=1)

# road quality
service = 'all_weather_roads'
df = svcs[svcs.s5fq0=='All-weather roads'].rename(columns={'s5fq4':f'dist_2_{service}'})[['hhid','s5fq6']]
tmp = pd.get_dummies(df.s5fq6).rename(columns={'Not satisfied':'road_unsatisfactory','Satisfied':'road_satisfactory'})
road_quality = pd.concat([df['hhid'],tmp],axis=1)

HH_service_access = reduce(lambda x, y: x.merge(y, on='hhid'), [road_quality,stage_quality,df_svcs])
HH_service_access


,hhid,road_unsatisfactory,road_satisfactory,stage_unsatisfactory,stage_satisfactory,time_2_food_market_or_shop,dist_2_food_market_or_shop,time_2_transport_stage,dist_2_transport_stage,time_2_all_weather_roads,...,time_2_health_centre,dist_2_health_centre,time_2_sector_office,dist_2_sector_office,time_2_Cellule_office,dist_2_Cellule_office,bicycle_access,motorcycle_access,vehicle_access,minibus_access
0,200001.0,0,1,0,1,30.0,2.0,30.0,2.0,5.0,...,30.0,2.0,15.0,1.0,7.0,0.30,0,0,0,0
1,200002.0,0,1,0,1,20.0,1.3,15.0,1.0,5.0,...,40.0,2.5,20.0,1.2,7.0,0.50,0,0,1,0
2,200003.0,0,1,0,1,30.0,2.0,15.0,1.0,5.0,...,20.0,1.3,15.0,1.0,7.0,0.50,0,0,0,0
3,200004.0,0,1,0,1,20.0,2.0,10.0,1.0,3.0,...,40.0,2.5,30.0,2.0,10.0,0.80,0,1,0,0
4,200005.0,0,1,1,0,15.0,1.5,10.0,1.0,3.0,...,15.0,1.0,20.0,3.0,5.0,0.40,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14575,214576.0,0,1,0,1,60.0,12.0,90.0,9.0,1.0,...,30.0,3.0,35.0,3.5,2.0,0.20,1,0,0,0
14576,214577.0,0,1,1,0,60.0,12.0,90.0,9.0,1.0,...,30.0,3.0,35.0,3.5,2.0,0.20,1,0,0,0
14577,214578.0,1,0,0,0,60.0,12.0,NaN,NaN,1.0,...,30.0,3.0,35.0,3.5,2.0,0.15,0,0,0,0
14578,214579.0,1,0,1,0,115.0,11.5,90.0,9.0,1.0,...,30.0,3.0,35.0,3.5,1.0,0.10,1,0,0,0


### Save files

In [17]:
%%time
df   = appliance_count
dir_ = '/home/qsel/Desktop/My_Drive/.shared/rwanda_drive/elec_consumption/data/processed/'
# dir_ = '/Users/joelmugyenyi/Google Drive/My Drive/rwanda_drive/elec_consumption/data/processed/'
file = 'appliance_count'

df.to_csv(dir_+file+'.csv',index=False)

CPU times: user 123 ms, sys: 5.36 ms, total: 128 ms
Wall time: 11.7 s


In [19]:
appliance_presence_HH.columns

Index(['hhid', 'Radio_present', 'TV_present', 'satellite_dish_present',
       'dvd_player_present', 'decoder_present', 'music_syst_present',
       'cooker_present', 'laundry_mach_present', 'electric_fan_present',
       'fridge_present', 'generator_as_appliance_present', 'hotplate_present',
       'power_stabilizer_present', 'still_camera_present',
       'video_camera_present', 'printer_present', 'computer_present',
       'Mobile_phones_present', 'smart_phones_present', 'tier_level',
       'tier_level_numeric', 'tier_0', 'tier_1', 'tier_2', 'tier_3', 'tier_4'],
      dtype='object')

### END